# Step 2: EDA & Data Cleaning

Explore missing patterns, default rates by segment, and clean data for modeling.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from spark_utils import get_spark_session, load_parquet, save_parquet
from data_cleaning import encode_target, compute_missing_rate, create_lgd_column, create_ead_column, handle_missing_numeric, handle_missing_categorical

In [ ]:
SAMPLE_MODE = True

## 2.1 Load Data

In [ ]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/accepted_raw.parquet')

## 2.2 Missing Value Analysis

In [ ]:
missing = compute_missing_rate(df)
missing_df = pd.DataFrame(missing.items(), columns=['column', 'missing_rate'])
missing_df[missing_df['missing_rate'] > 0].head(20)

In [ ]:
# Visualize top 20 missing columns
top_missing = missing_df[missing_df['missing_rate'] > 0].head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_missing['column'], top_missing['missing_rate'])
ax.set_xlabel('Missing Rate')
ax.set_title('Top 20 Columns by Missing Rate')
plt.tight_layout()
plt.show()

## 2.3 Target Variable Encoding

In [ ]:
df = encode_target(df)
df.groupBy('default_flag').count().show()

## 2.4 Default Rate by Segment

In [ ]:
# Default rate by grade
from pyspark.sql import functions as F

grade_default = df.groupBy('grade').agg(
    F.mean('default_flag').alias('default_rate'),
    F.count('*').alias('count')
).orderBy('grade')
grade_default.show()

In [ ]:
# Default rate by application type
df.groupBy('application_type').agg(
    F.mean('default_flag').alias('default_rate'),
    F.count('*').alias('count')
).show()

## 2.5 LGD Distribution

In [ ]:
df = create_lgd_column(df)
df = create_ead_column(df)

# LGD histogram (zero-inflated / bimodal)
lgd_pd = df.filter(F.col('default_flag') == 1).select('lgd').toPandas()
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(lgd_pd['lgd'].dropna(), bins=50, edgecolor='black')
ax.set_xlabel('LGD')
ax.set_title('LGD Distribution (Defaulted Loans)')
plt.tight_layout()
plt.show()

## 2.6 Handle Missing Values & Save

In [ ]:
# TODO: specify numeric and categorical columns for imputation
# numeric_cols = ['annual_inc', 'dti', 'revol_util', ...]
# categorical_cols = ['emp_title', 'title', ...]
# df = handle_missing_numeric(df, numeric_cols)
# df = handle_missing_categorical(df, categorical_cols)

save_parquet(df, '/home/jovyan/work/data/parquet/accepted_cleaned.parquet')
print('Cleaned data saved.')